# Deep Hedging AI (JAX port)
### Recurrent Network: impact of using recurrent networks on hedging barrier options. In this example, Deep Hedging does not learn anything - which is superior to using Black & Scholes.

Hans Buehler, January 2023

Port of `notebooks/trainer-recurrent-barrier_smalldt.ipynb` by Hans Buehler, January 2023.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import math
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from IPython.display import display, Markdown
from deephedging.world import WorldConfig, simulate
from deephedging.model import AgentConfig, GymConfig, oce_utility
from deephedging.train import TrainConfig, fit
from deephedging.plotting import plot_training
from deephedging.stats import mean_bins
from deephedging.fd import bs_fd

# DH_EPOCHS overrides every epoch count for quick smoke runs; DH_CHECKPOINTS="" turns checkpoints off.
epochs = lambda n: int(os.environ.get("DH_EPOCHS") or n)
CHECKPOINTS = os.environ.get("DH_CHECKPOINTS", "checkpoints") or None

In [ ]:
nBins = 200
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

strike  = 1.
barrier = 0.95
def payoff_def( spots ):
    notko      = np.cumprod( np.where( spots[:,:] > barrier, 1., 0. ), axis=1 )
    terminal   = np.maximum( 1. - spots[:,-1], 0. )  # put
    return dict( payoff=-terminal*notko[:,-1],
                 features=notko[:,:-1] )
payoff_def.name = "KO Put k=%g, b=%g" % (strike, barrier)

world_cfg = WorldConfig(steps=20, samples=20000, drift=0., cost_s=0., black_scholes=True, payoff=payoff_def)
agent_ref = AgentConfig(depth=5, width=50, activation="softplus", classic=1, aggregate=1, past_repr=1, event=1,
                        features=("price", "delta", "time_left", "payoff_features"))
gym_ref   = GymConfig(agent=agent_ref, utility="cvar", lmbda=1.01)
train_cfg = TrainConfig(epochs=epochs(250), batch_size=32, learning_rate=0.001, clipvalue=0.9, global_clipnorm=1.)

# create world
world      = simulate(world_cfg)
val_world  = world.clone(samples=world.nSamples // 2, seed=1)

In [ ]:
display(Markdown("## Plotting payoff " + payoff_def.name))

payoff = world.market["payoff"]
spots  = world.spot_all
notko  = world.features["payoff_features"][:, :, 0]
ref    = spots[:,-1]
ixs    = np.argsort(ref)
nko_s  = notko[ixs,-1]
ref_s  = ref[ixs]
pyo_s  = payoff[ixs]

fig, ax = plt.subplots(figsize=(10,5))
ax.set_title("%s" % payoff_def.name)

for color, label, mask in zip(colors, ["payoff", "not KO'd", "has KO'd"], [np.ones(len(ref_s), dtype=bool), nko_s > 0., nko_s < 1.]):
    xm   = mean_bins( ref_s[mask], bins=nBins, weights=None, return_std = False )
    m, s = mean_bins( pyo_s[mask], bins=nBins, weights=None, return_std = True )
    ax.plot( xm, m, color=color, label=label)
    ax.fill_between( xm, m-s,m+s, color=color,alpha=0.2)

ax.legend()
plt.show()

In [ ]:
# Create test configs
# -------------------
configs = dict(
    # not recurrent
    none    = replace(agent_ref, classic=0, aggregate=0, past_repr=0, event=0),
    # digital states
    digital = replace(agent_ref, classic=0, aggregate=0, past_repr=5, event=5),
)

# train
# ----
fits = {}

for k, agent in configs.items():
    display(Markdown("## Hedging a %s in recurrent '%s' mode" % (payoff_def.name, k)))
    gym = GymConfig(agent=agent, utility=gym_ref.utility, lmbda=gym_ref.lmbda)
    fits[k] = fit(gym, train_cfg, world, val_world, log_every=25, checkpoint_dir=CHECKPOINTS)
    plot_training(fits[k].result, fits[k].train, fits[k].val, world, val_world)
    plt.show()

## Black & Scholes reference and per-step deltas

All 20 time steps are shown.

In [ ]:
spots      = world.spot_all
hedges     = world.market["hedges"][:,:,0]
nSteps     = world.nSteps
nSamples   = world.nSamples
nBins      = 200
fdstd      = 5.
fdgrid     = 501
times      = world.timeline
vol        = world_cfg.rvol

# collect actions
actions = { k: fits[k].train['actions'][:,:,0] for k in fits }
deltas  = { k: np.cumsum( actions[k], axis=1) for k in fits }
pnls    = { k: np.zeros((nSamples,)) for k in fits }
pnls['blackscholes'] = np.zeros((nSamples,))
prev_deltas_t = { k: 0. for k in pnls }

keys_sorted = ['blackscholes'] + list( fits.keys() )

# FD pricer
# ---------
def fd_payoff( X, F, t ):
    F     = - np.maximum( strike - X, 0.) if F is None else F
    return np.where( X > barrier, F, 0. )
fd_spots = np.exp( np.linspace(-fdstd,+fdstd,1+2*(fdgrid//2),endpoint=True) * vol * times[-1] )
fd = bs_fd( spots=[fd_spots]*(nSteps+1), times=times, payoff=fd_payoff, vol=vol, cn_factor='implicit')

# plot hedging
# ------------
display(Markdown("## %s Hedging" % payoff_def.name))
shown = list(range(0, nSteps, 1))
ncols = 5
nrows = math.ceil(len(shown) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False)
axes = axes.ravel()
for ax in axes[len(shown):]:
    ax.axis("off")

for j in range(nSteps):
    # sort by spot at j, and compute BS reference
    spot_t    = spots[:,j]
    hedges_t  = hedges[:,j]   # S(T)-S(t_j)
    dt        = times[j+1] - times[j]

    deltas_t  = { k: deltas[k][:,j] for k in deltas }
    deltas_t['blackscholes'] \
              = - fd[j].bump_delta( spot_t, vol * math.sqrt(dt) )   # fd prices the payoff Z itself; its hedge is -delta
    trades_t  = { k: deltas_t[k] - prev_deltas_t[k] for k in pnls }
    pnls      = { k: pnls[k] + trades_t[k] * hedges_t for k in pnls }
    prev_deltas_t = dict(deltas_t)

    if j not in shown:
        continue

    # plot
    ax_delta  = axes[shown.index(j)]
    ixs       = np.argsort(spot_t)
    spot_t    = spot_t[ixs]
    deltas_t  = { k: deltas_t[k][ixs] for k in deltas_t }

    x_mean    = mean_bins( spot_t, bins=nBins, weights=None, return_std = False )
    for color, k in zip(colors, keys_sorted):
        mean, std = mean_bins( deltas_t[k], bins=nBins, weights=None, return_std = True )
        ax_delta.plot( x_mean, mean, label=k, color=color)
        ax_delta.fill_between( x_mean, mean-std, mean+std, color=color, alpha=0.1 )

    ax_delta.set_title("Deltas time step %d (%g)" % (j, times[j]))
    ax_delta.legend()

fig.tight_layout()
plt.show()

In [ ]:
display(Markdown("## Terminal %s Profiles" % payoff_def.name))

spots    = world.spot_all
payoff   = world.market["payoff"]
notko    = world.features["payoff_features"][:, -1, 0]
payoff   = payoff - np.mean( payoff )

spot_ret = spots[:,-1]
ixs      = np.argsort(spot_ret)
spot_ret = spot_ret[ixs]
pnls     = { k : pnls[k][ixs] for k in pnls }
payoff   = payoff[ixs]
notko    = notko[ixs]

panels = [("Terminal Payoff", np.ones(len(notko), dtype=bool), pnls, keys_sorted),
          ("Terminal Payoff\nNot KO'd", notko > 0., pnls, keys_sorted),
          ("Terminal Payoff\nKO'd", notko < 1., pnls, keys_sorted)]
fig, axes = plt.subplots(1, len(panels), figsize=(10*len(panels), 5))
_min, _max = None, None
for ax, (title, mask, series, keys) in zip(axes, panels):
    spot_mean = mean_bins( spot_ret[mask], bins=nBins, weights=None, return_std = False )
    payoff_mean, payoff_std = mean_bins( payoff[mask], bins=nBins, weights=None, return_std = True )

    ax.set_title(title)
    ax.set_xlabel("Spot")
    ax.plot( spot_mean, -payoff_mean, label="-payoff", color=colors[0])
    ax.fill_between( spot_mean, -payoff_mean-payoff_std, -payoff_mean+payoff_std , color=colors[0], alpha=0.1 )
    if _min is None:
        _min = np.min(-payoff_mean-payoff_std)
        _max = np.max(-payoff_mean+payoff_std)
        first = True
    else:
        first = False
    for color, k in zip(colors[1:], keys):
        mean, std = mean_bins( series[k][mask], bins=nBins, weights=None, return_std = True )
        ax.plot( spot_mean, mean, label=k, color=color )
        ax.fill_between( spot_mean, mean-std, mean+std, color=color, alpha=0.1 )
        if first:
            _min = min(np.min(mean),_min)
            _max = max(np.max(mean),_max)
    ax.legend()
    if first:
        _min -= max(_max - _min, 0.001)*0.005

for ax in axes:
    ax.set_ylim(_min,_max)
fig.tight_layout()
plt.show()

In [ ]:
display(Markdown("## Utilities for %s" % payoff_def.name))

utilities = ['cvar', 'exp2', 'exp', 'vicky', 'quad' ]
lambdas  = np.exp(np.linspace( -2.,2., 21, endpoint=True) )

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
axes[-1].axis("off")

for ax, util in zip(axes, utilities):
    ax.set_title(util)
    ax.set_xlabel("Log-Lambda")

    for color, k in zip(colors, pnls):
        pnl = pnls[k]
        us  = [ oce_utility(util, lmnda, pnl ) for lmnda in lambdas ]
        ax.plot( np.log(lambdas), us, color=color, label=k )

    ax.legend()

fig.tight_layout()
plt.show()